# 08 — Experiment 5c

**Same scope as 5a/5b.** 5a crashed at step 1 with `vectorized_gather_kernel: index out of bounds`; 5b's dequantize fix was a no-op (embedding was already fp16) and crashed the same way.

**5c hypothesis:** `resize_token_embeddings` grew `embed_tokens` (262145 ✓, confirmed in 5b cell 7 output) but did NOT re-tie `lm_head`. Labels contain token ID 262144 (`[civicinsight-v1]`) → cross-entropy gathers against an undersized lm_head → async CUDA OOB surfaces as a misleading vision-tower trace.

**5c fix:** call `model.tie_weights()` after resize + explicit shape assert on lm_head.

**Knobs (unchanged):** special token + `repetition_penalty=1.2` + `no_repeat_ngram_size=4`. Vision frozen. r=16, α=32, 5 epochs, LR 2e-4.

In [ ]:
%%capture
!pip install unsloth
!pip install pillow==11.3.0

In [ ]:
# imports

from unsloth import FastVisionModel          # MUST be first
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
import torch
import os
import json
from PIL import Image
from huggingface_hub import login, snapshot_download
import time

# this line came from 01 zeroshot tests
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Logged in via Kaggle Secrets")
except Exception:
    print("Could not find HF_TOKEN in Kaggle Secrets. Set it up or login manually.")

In [ ]:
# one-time execute to download the model
path = snapshot_download(
    repo_id="unsloth/gemma-4-e4b-it",
    local_dir="/kaggle/working/gemma4-unsloth",
    ignore_patterns=["*.md"],
)
print(f"Downloaded to: {path}")


In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    "/kaggle/working/gemma4-unsloth",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
print("Model successfully loaded")
print(f"Vocab size BEFORE add_tokens: {len(tokenizer.tokenizer)}")

# ── Special-token registration — BEFORE get_peft_model ──
# 5c fix: after resize, force tie_weights() so lm_head picks up the new row.
# In 5b we confirmed embed_tokens grew (262145 rows) but lm_head likely didn't,
# causing cross-entropy OOB when labels contain the new token ID 262144.
num_added = tokenizer.tokenizer.add_tokens(["[civicinsight-v1]"], special_tokens=True)
model.resize_token_embeddings(len(tokenizer.tokenizer), mean_resizing=False)

emb_rows = model.get_input_embeddings().weight.shape[0]
lm_rows_before = model.get_output_embeddings().weight.shape[0]
print(f"BEFORE tie_weights — embed_tokens: {emb_rows} rows, lm_head: {lm_rows_before} rows")

model.tie_weights()
lm_rows_after = model.get_output_embeddings().weight.shape[0]
print(f"AFTER  tie_weights — embed_tokens: {emb_rows} rows, lm_head: {lm_rows_after} rows")

assert emb_rows == len(tokenizer.tokenizer), "embed_tokens rows != vocab size"
assert lm_rows_after == len(tokenizer.tokenizer), \
    f"lm_head still {lm_rows_after} rows (expected {len(tokenizer.tokenizer)}) — tie_weights did not propagate. Need explicit lm_head replacement."

new_id = tokenizer.tokenizer.convert_tokens_to_ids("[civicinsight-v1]")
print(f"New token ID: {new_id} (expected: {len(tokenizer.tokenizer) - 1})")
print(f"Added {num_added} special token(s). Vocab size AFTER: {len(tokenizer.tokenizer)}")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# attach LoRA
model = FastVisionModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    finetune_vision_layers=False,
)

print(model.print_trainable_parameters())


In [ ]:
# load our civicinsight dataset
DATASET_JSON = "/kaggle/input/datasets/shahfazalmohammed/civicinsight-dataset/dataset.marked.json"
KAGGLE_IMAGE_DIR = "/kaggle/input/datasets/shahfazalmohammed/civicinsight-dataset/standardized/standardized"

# build the message
# Input: PIL image
#        aria_label
#        prompt
# Output: message built as prompt with inputs
def to_conversation(img, aria_label, prompt):
    return {
        "messages": [
            {
                "role": "user", # one role as user with image and prompt
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": prompt}
                ],
            },
            {
                "role": "assistant", # one role as assistant on what to train on
                "content": [
                     {"type": "text", "text": aria_label},
                ]              
            }
        ]
    }
    
with open(DATASET_JSON) as f:
    dataset = json.load(f)

conversations = []
for record in dataset:
    # fix the path name from the dataset to match the Kaggle dataset dir format
    img_path = os.path.join(KAGGLE_IMAGE_DIR, os.path.basename(record["image"]))
    # convert to PIL
    img = Image.open(img_path)
    # convert to conversation, append to conversations.
    conversations.append(to_conversation(img, record["aria_label"], record["prompt"]))

# sanity check, always
print(f"Loaded {len(conversations)} conversations")
print(conversations[0]["messages"][0]["content"][0])  # should print the prompt text


In [ ]:
# ============================================================
# BEFORE-TRAINING baseline — capture raw Gemma output on a training image
# This is the 'before' half of the before/after comparison.
# ============================================================
model.eval()
BASELINE_TRAIN_IDX = 5  # same index used in post-training cell
message_pre = conversations[BASELINE_TRAIN_IDX]["messages"][:1]
inputs = tokenizer.apply_chat_template(
    message_pre,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)
start = time.time()
outputs = model.generate(**inputs, max_new_tokens=400, repetition_penalty=1.2, no_repeat_ngram_size=4)
elapsed = time.time() - start
baseline_pre_train = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"=== BEFORE training | training idx {BASELINE_TRAIN_IDX} | {elapsed:.1f}s ===")
print(baseline_pre_train)
model.train()  # back to train mode before trainer.train()

In [ ]:
# epoch run test
trainer = SFTTrainer(
    model=model,                                          # the LoRA-wrapped Gemma 4
    tokenizer=tokenizer,                                  # handles text + image tokenization
    data_collator=UnslothVisionDataCollator(model, tokenizer),  # batches image+text together (vision-specific, replaces default)
    train_dataset=conversations,                          # full 50 examples; max_steps stops early
    args=SFTConfig(
        per_device_train_batch_size=1,                   # 1 example per forward pass (GPU memory constraint)
        gradient_accumulation_steps=4,                   # accumulate 4 steps before updating weights = effective batch of 4
        num_train_epochs=5,                              # run our standar 3-epoch test.
        learning_rate=2e-4,                              # standard LoRA starting LR
        output_dir="./test_output",                      # checkpoint save location (kaggle/working/)
        max_seq_length=2048,                             # max tokens per example (image + text combined)
        dataset_text_field="",                           # tells SFT not to look for a text column (we handle format ourselves)
        dataset_kwargs={"skip_prepare_dataset": True},   # skip HuggingFace auto-formatting (our format is already correct)
    )
)

print(f"GPU before training: {torch.cuda.memory_allocated()/1e9:.1f} GB")
trainer.train()
print("Training ran without crash!")
print(f"GPU after training: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# ============================================================
# AFTER-TRAINING on same training image — 'after' half of comparison
# If this is ~identical to the gold aria_label, training memorized (good).
# If it's still adjective-y like Exp 3, training didn't take.
# ============================================================
model.eval()
message = conversations[BASELINE_TRAIN_IDX]["messages"][:1]
inputs = tokenizer.apply_chat_template(
    message,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)
start = time.time()
outputs = model.generate(**inputs, max_new_tokens=400, repetition_penalty=1.2, no_repeat_ngram_size=4)
elapsed = time.time() - start
baseline_post_train = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"=== AFTER training | training idx {BASELINE_TRAIN_IDX} | {elapsed:.1f}s ===")
print(baseline_post_train)
print()
print("=== GOLD aria_label for reference ===")
print(dataset[BASELINE_TRAIN_IDX]["aria_label"])

In [ ]:
# ============================================================
# Held-out sweep — run inference on EVERY image in the test dir
# Upload 3-4 varied chart types (bar, line, map, scatter) to civicinsight-test
# before running this cell for meaningful generalization signal.
# ============================================================
KAGGLE_TEST_IMG_DIR = "/kaggle/input/datasets/shahfazalmohammed/civicinsight-test"
prompt = "Generate an aria-label for this data visualization image."

test_images = sorted([f for f in os.listdir(KAGGLE_TEST_IMG_DIR) if f.endswith(('.png', '.jpg', '.jpeg'))])
print(f"Found {len(test_images)} held-out images\n")

held_out_results = {}
for img_name in test_images:
    img_path = os.path.join(KAGGLE_TEST_IMG_DIR, img_name)
    img = Image.open(img_path)
    message = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": prompt}]}]
    inputs = tokenizer.apply_chat_template(
        message,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)
    start = time.time()
    outputs = model.generate(**inputs, max_new_tokens=600, repetition_penalty=1.2, no_repeat_ngram_size=4)
    elapsed = time.time() - start
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    held_out_results[img_name] = decoded
    print(f"{'='*70}")
    print(f"IMAGE: {img_name} | {elapsed:.1f}s")
    print(f"{'='*70}")
    print(decoded)
    print()

In [ ]:
# ============================================================
# Min Viable Bar — automated imprint score on held-out outputs
# ============================================================
MARKER = "[civicinsight-v1]"
SLOT_OPENERS = ("This line", "This bar", "This stacked", "This box", "This scatter",
                "This choropleth", "This hexagonal", "This panel", "This small multiples",
                "This untitled", "This area", "This pie", "This gauge", "This table")
BANNED_ADJECTIVES = ("around ", "approximately", "roughly ", "about ", "appears to",
                     "notably", "dramatically", "rises steadily", "drops steadily",
                     "significantly", "suggesting")

def score(output):
    # strip the chat template wrapping to get just the assistant response
    assistant_part = output.split("model\n")[-1] if "model\n" in output else output
    assistant_part = assistant_part.strip()
    return {
        "has_marker": MARKER in assistant_part,
        "opens_with_slot": any(assistant_part.lstrip().startswith(MARKER + " " + s) or assistant_part.lstrip().startswith(s) for s in SLOT_OPENERS),
        "banned_adjectives_found": [a.strip() for a in BANNED_ADJECTIVES if a in assistant_part.lower()],
    }

print(f"{'image':<45} {'marker':<8} {'slot-open':<10} {'banned-hits'}")
print("-" * 100)
for img_name, out in held_out_results.items():
    s = score(out)
    banned = ",".join(s["banned_adjectives_found"]) or "none"
    print(f"{img_name:<45} {str(s['has_marker']):<8} {str(s['opens_with_slot']):<10} {banned}")

print()
# aggregate
n = len(held_out_results)
marker_rate = sum(1 for o in held_out_results.values() if score(o)['has_marker']) / n if n else 0
slot_rate = sum(1 for o in held_out_results.values() if score(o)['opens_with_slot']) / n if n else 0
no_adj_rate = sum(1 for o in held_out_results.values() if not score(o)['banned_adjectives_found']) / n if n else 0
print(f"Summary across {n} held-out images:")
print(f"  marker appears:          {marker_rate:.0%}")
print(f"  opens with slot pattern: {slot_rate:.0%}")
print(f"  no banned adjectives:    {no_adj_rate:.0%}")